# OpenPlaque — Vendor Q3D Proximal Origin Audit v1

This experiment audits the vendor-derived coronary radial curved Q3D series:

- **1035 / folder 32218:** RCA Curved Range Radial Q3D(MT) — positive control
- **1039 / folder 32219:** CX Curved Range Radial Q3D(MT)
- **1043 / folder 32220:** LAD Curved Range Radial Q3D(MT)

Each contains 24 derived 512×512 images.

**Critical rule:** the 24 files are treated as radial curved views around one vessel, **not as 24 physical slices of a 3-D volume**.

The experiment first inventories standard/private DICOM geometry and source-image references. It then performs a pixel-space proximal-origin audit calibrated to the RCA series. No clinical LM/LCX/OM identity is promoted and the frozen master is not modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paths and reuse controls — immediately after Drive mount
DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
DICOM_ROOT = "/content/drive/MyDrive/CCTA/DICOM/3221"
OUT = f"{DRIVE_ROOT}/Vendor_Q3D_Proximal_Origin_Audit_v1"

print("DICOM root:", DICOM_ROOT)
print("Output:", OUT)
print("RCA Q3D:", f"{DICOM_ROOT}/32218")
print("CX Q3D:", f"{DICOM_ROOT}/32219")
print("LAD Q3D:", f"{DICOM_ROOT}/32220")

In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "b5856aad0e55803c23a26082c8e570d5870732b7"
OPENPLAQUE_BRANCH = "vendor-q3d-proximal-origin-audit-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q pylibjpeg pylibjpeg-libjpeg
%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())

In [ ]:
# Synthetic tests before study data
from openplaque.vendor_q3d_proximal_origin_audit_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_vendor_q3d_proximal_origin_audit_v1.py

In [ ]:
from openplaque.vendor_q3d_proximal_origin_audit_v1 import run

for folder in ["32218","32219","32220","32210"]:
    p = Path(DICOM_ROOT)/folder
    if not p.is_dir():
        raise FileNotFoundError(p)

summary = run(
    dicom_root=DICOM_ROOT,
    drive_root=DRIVE_ROOT,
    output_dir=OUT,
)

print(json.dumps(summary["decision"], indent=2))
print("\nStatus:", summary["status"])

In [ ]:
# Primary quantitative review
import pandas as pd
from IPython.display import display

sig = pd.read_csv(Path(OUT)/"q3d_origin_signatures.csv")
views = pd.read_csv(Path(OUT)/"q3d_view_metrics.csv")

print("RCA-calibrated origin signatures:")
display(sig)

for vessel in ["RCA","LAD","CX"]:
    g = views[views.vessel==vessel]
    side = summary["decision"]["rca_proximal_side"]
    print(f"\n{vessel}: strongest {side}-side Q3D views")
    display(
        g.nlargest(8, f"{side}_origin_score")[
            ["view_index","detected_axis","axis_agrees_with_renderer",
             f"{side}_origin_score",
             f"{side}_expansion_ratio",
             f"{side}_component_area_fraction",
             f"{side}_component_max_width_px"]
        ]
    )

In [ ]:
# DICOM geometry / provenance review
geom = pd.read_csv(Path(OUT)/"q3d_geometry_inventory.csv")
refs = pd.read_csv(Path(OUT)/"q3d_source_reference_mapping.csv")
tags = pd.read_csv(Path(OUT)/"q3d_tag_inventory.csv")
private = pd.read_csv(Path(OUT)/"q3d_private_tag_inventory.csv")

print("Standard spatial geometry by vessel:")
display(
    geom.groupby("vessel", as_index=False).agg(
        n_images=("file","count"),
        standard_plane_geometry_images=("has_standard_plane_geometry","sum"),
        unique_frame_uids=("frame_of_reference_uid","nunique"),
    )
)

print("Series-7 source-reference linkage:")
display(
    refs.groupby("vessel", as_index=False).agg(
        total_referenced_sops=("referenced_sop_count","sum"),
        matched_series7_sops=("matched_series7_sop_count","sum"),
    )
)

print("Private tags whose values vary across the 24 radial views:")
display(
    private[private.n_unique_values>1][
        ["vessel","path","tag","keyword","name","vr","n_unique_values","example_values"]
    ].head(80)
)

In [ ]:
# Automatic image QC
from IPython.display import Image, display, HTML

display(Image(filename=str(Path(OUT)/"01_q3d_proximal_width_profiles.png"), width=1000))
for vessel in ["RCA","LAD","CX"]:
    p = Path(OUT)/f"QC_{vessel}_q3d_informative_views.png"
    print(p.name)
    display(Image(filename=str(p), width=1200))

report = Path(OUT)/"OPENPLAQUE_VENDOR_Q3D_PROXIMAL_ORIGIN_AUDIT_V1_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Verify deliverables
expected = [
    "run_state.json",
    "summary.json",
    "decision.json",
    "q3d_geometry_inventory.csv",
    "q3d_tag_inventory.csv",
    "q3d_private_tag_inventory.csv",
    "q3d_source_reference_mapping.csv",
    "q3d_view_metrics.csv",
    "q3d_origin_signatures.csv",
    "01_q3d_proximal_width_profiles.png",
    "QC_RCA_q3d_informative_views.png",
    "QC_LAD_q3d_informative_views.png",
    "QC_CX_q3d_informative_views.png",
    "OPENPLAQUE_VENDOR_Q3D_PROXIMAL_ORIGIN_AUDIT_V1_REPORT.html",
    "OPENPLAQUE_VENDOR_Q3D_PROXIMAL_ORIGIN_AUDIT_V1_RESULTS.zip",
]
missing=[x for x in expected if not (Path(OUT)/x).exists()]
if missing:
    raise RuntimeError("Missing outputs: "+str(missing))

state=json.loads((Path(OUT)/"run_state.json").read_text())
if state.get("status")!="COMPLETE":
    raise RuntimeError("Run state not COMPLETE: "+str(state))

print("COMPLETE")
for x in expected:
    print(Path(OUT)/x)